<a href="https://colab.research.google.com/github/Pavan2412/Inflation-prediction-with-ARIMA-and-LSTM-using-CPI/blob/main/inflation_predictor_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 Inflation Predictor: ARIMA vs ARIMAX vs LSTM vs Random Walk

**A research-grade macro/markets forecasting comparison, built to sell-side Global
Markets / quant-research standards.**

> ⚠️ **Data note (read first):** This environment's network access is restricted to a
> small whitelist of software-package domains (PyPI, npm, GitHub, etc.) — **FRED,
> Yahoo Finance, and the Cleveland Fed / NY Fed nowcast pages are all unreachable**
> from here (confirmed: both return HTTP 403 via the network proxy). Rather than
> silently fall back to something misleading, this notebook uses a **synthetic data-
> generating process (DGP)** calibrated to realistic macro/market statistical
> properties (mean-reverting breakeven inflation, persistent CPI YoY, GBM-style asset
> prices with realistic cross-correlations). Every series is clearly labeled
> `_synthetic`. **Section 2** contains the exact swap-in code to replace the synthetic
> pull with live `pandas_datareader`/`fredapi`/`yfinance` calls — run that locally with
> open internet and the rest of the notebook is unchanged.


## 1. 📦 Imports & Configuration

In [ ]:

import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
!pip install pmdarima
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from pmdarima import auto_arima

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.float_format', '{:.4f}'.format)
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"   pandas : {pd.__version__}   numpy: {np.__version__}   tensorflow: {tf.__version__}")


## 2. 📥 Data Generation (Synthetic Macro/Markets Proxy)

**Target variable choice — why breakeven inflation, not the TIP ETF price:**
TIP's *price* is a function of real yields and duration, not inflation expectations
directly — a rate-hike scare can crush TIP price even while inflation expectations are
flat or rising. **10Y breakeven inflation (`T10YIE` = `DGS10 − DFII10`)** is the
market's actual priced-in inflation expectation and is the standard Street proxy. We
also carry **CPI YoY** as a secondary, slower-moving "ground truth" comparison target.

**Live-data swap-in (run locally with internet):**
```python
import pandas_datareader.data as web
breakeven = web.DataReader('T10YIE', 'fred', start, end)          # primary target
cpi = web.DataReader('CPIAUCSL', 'fred', start, end).pct_change(12) * 100  # YoY, secondary
import yfinance as yf
exog = yf.download(['GLD','USO','DX-Y.NYB','^TNX'], start=start, end=end, auto_adjust=True)['Close']
```


### 📥 2.1 Live Data Integration (FRED API)
To use the live data, please ensure you have added your FRED API key to the Colab Secrets (the 🔑 icon on the left) with the name `FRED_API_KEY`.

In [ ]:
import pandas_datareader.data as web
from google.colab import userdata
import yfinance as yf

try:
    # Set FRED API Key from Colab Secrets
    os.environ['FRED_API_KEY'] = userdata.get('FRED_API_KEY')

    # Define dates
    end = datetime.today()
    start = end - timedelta(days=1200) # Get ~4 years of data

    print("⏳ Fetching live data from FRED and Yahoo Finance...")

    # 1. Primary Target: 10Y Breakeven Inflation
    breakeven_live = web.DataReader('T10YIE', 'fred', start, end)

    # 2. Secondary Target: CPI YoY
    cpi_live = web.DataReader('CPIAUCSL', 'fred', start, end).pct_change(12) * 100

    # 3. Exogenous Drivers (Gold, Oil, Dollar Index, 10Y Yield)
    exog_live = yf.download(['GLD','USO','DX-Y.NYB','^TNX'], start=start, end=end, auto_adjust=True)['Close']

    # Merge and Align
    data_live = pd.concat([breakeven_live, cpi_live, exog_live], axis=1).dropna()
    data_live.columns = ['Breakeven', 'CPI_YoY', 'Dollar_Index', 'GLD', 'TNX', 'USO']

    # Reorder to match previous notebook structure
    data = data_live[['Breakeven', 'CPI_YoY', 'GLD', 'USO', 'Dollar_Index', 'TNX']]

    print(f"✅ Live data loaded successfully! Shape: {data.shape}")
    display(data.tail())

except Exception as e:
    print(f"❌ Error fetching live data: {e}")
    print("Ensure 'FRED_API_KEY' is set in your secrets and you have internet access.")

## 3. 🔍 Exploratory Data Analysis

In [ ]:

print(data.describe().T)


In [ ]:

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(data.index, data['Breakeven'], color='#d62728', lw=1.3, label='10Y Breakeven Inflation')
axes[0].plot(data.index, data['CPI_YoY'], color='#1f77b4', lw=1.1, alpha=0.8, label='CPI YoY (synthetic)')
axes[0].set_ylabel('%'); axes[0].legend(); axes[0].set_title('Primary & Secondary Inflation Targets')

norm = data[['GLD','USO','Dollar_Index','TNX']] / data[['GLD','USO','Dollar_Index','TNX']].iloc[0] * 100
for col in norm.columns:
    axes[1].plot(data.index, norm[col], lw=1.1, label=col)
axes[1].set_ylabel('Indexed to 100'); axes[1].legend(); axes[1].set_title('Exogenous Drivers (Normalized)')
plt.tight_layout(); plt.show()


In [ ]:

plt.figure(figsize=(7, 5.5))
sns.heatmap(data.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Matrix — Target & Drivers')
plt.tight_layout(); plt.show()


## 4. 📐 Stationarity Testing & Decomposition

In [ ]:

def adf_test(series, name='Series'):
    result = adfuller(series.dropna(), autolag='AIC')
    stat, pval, lags, _, crit, _ = result
    status = '✅ Stationary' if pval < 0.05 else '❌ Non-Stationary'
    print(f"{status}  |  {name:<16} ADF={stat:8.4f}  p={pval:.6f}  Lags={lags}")
    return pval < 0.05

print("── Levels ─────────────────────────────────────")
for col in data.columns:
    adf_test(data[col], col)

print("\n── First Differences ──────────────────────────")
diff_data = data.diff().dropna()
for col in diff_data.columns:
    adf_test(diff_data[col], col)


In [ ]:

decomp = seasonal_decompose(data['Breakeven'], model='additive', period=63)  # ~quarterly
fig = decomp.plot()
fig.set_size_inches(11, 7)
plt.tight_layout(); plt.show()


## 5. 📉 ACF & PACF — ARIMA Order Selection

In [ ]:

be_diff = data['Breakeven'].diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(be_diff, lags=30, ax=axes[0]); axes[0].set_title('ACF — Δ Breakeven')
plot_pacf(be_diff, lags=30, ax=axes[1]); axes[1].set_title('PACF — Δ Breakeven')
plt.tight_layout(); plt.show()


## 6. ⚖️ Walk-Forward Backtest Setup (shared across all models)

To make ARIMA, ARIMAX, LSTM and the Random Walk benchmark **directly comparable**,
every model is scored over the **exact same rolling-origin test window**: an expanding
training window, re-forecasting one step ahead at every point in the test period.


In [ ]:

TARGET = 'Breakeven'
EXOG_COLS = ['GLD', 'USO', 'Dollar_Index', 'TNX']

split_idx = int(len(data) * 0.85)
train_df, test_df = data.iloc[:split_idx], data.iloc[split_idx:]
TEST_DATES = test_df.index

print(f"Training samples : {len(train_df)}  ({train_df.index[0].date()} → {train_df.index[-1].date()})")
print(f"Test samples     : {len(test_df)}   ({test_df.index[0].date()} → {test_df.index[-1].date()})")


## 7. 🎲 Benchmark 0 — Naive Random Walk

In [ ]:

y_true = test_df[TARGET].values
rw_preds = data[TARGET].shift(1).iloc[split_idx:].values   # "today predicts tomorrow"

def theils_u(y_true, y_pred, y_naive):
    num = np.sqrt(np.mean((y_pred - y_true) ** 2))
    den = np.sqrt(np.mean((y_naive - y_true) ** 2))
    return num / den

rw_rmse = np.sqrt(mean_squared_error(y_true, rw_preds))
rw_mae  = mean_absolute_error(y_true, rw_preds)
print(f"Random Walk  RMSE={rw_rmse:.4f}  MAE={rw_mae:.4f}   (Theil's U = 1.000 by definition)")


## 8. 🔢 Model 1 — ARIMA (univariate)

In [ ]:

print("🔍 Running auto_arima …")
arima_auto = auto_arima(
    train_df[TARGET], start_p=0, max_p=5, start_q=0, max_q=5, d=None,
    seasonal=False, information_criterion='aic', stepwise=True,
    suppress_warnings=True, error_action='ignore', trace=True,
)
print(f"\n✅ Best order: ARIMA{arima_auto.order}   AIC={arima_auto.aic():.4f}")


In [ ]:

p, d, q = arima_auto.order
history = list(train_df[TARGET])
arima_preds, arima_ci_lo, arima_ci_hi = [], [], []

for i in range(len(test_df)):
    fit = ARIMA(history, order=(p, d, q)).fit()
    fc = fit.get_forecast(steps=1)
    arima_preds.append(fc.predicted_mean[0])
    ci = fc.conf_int(alpha=0.05)
    arima_ci_lo.append(ci[0][0]); arima_ci_hi.append(ci[0][1])
    history.append(test_df[TARGET].iloc[i])
    if (i + 1) % 50 == 0:
        print(f"   … {i+1}/{len(test_df)} steps")

arima_preds = np.array(arima_preds)
arima_ci_lo, arima_ci_hi = np.array(arima_ci_lo), np.array(arima_ci_hi)
arima_rmse = np.sqrt(mean_squared_error(y_true, arima_preds))
arima_mae  = mean_absolute_error(y_true, arima_preds)
arima_u    = theils_u(y_true, arima_preds, rw_preds)
print(f"\n✅ ARIMA  RMSE={arima_rmse:.4f}  MAE={arima_mae:.4f}  Theil's U={arima_u:.4f}")


## 9. 🔢 Model 2 — ARIMAX (fair multivariate benchmark)

In [ ]:

history_y = list(train_df[TARGET])
history_x = train_df[EXOG_COLS].values.tolist()
arimax_preds = []

for i in range(len(test_df)):
    fit = ARIMA(history_y, order=(p, d, q), exog=np.array(history_x)).fit()
    next_x = test_df[EXOG_COLS].iloc[[i]].values
    fc = fit.get_forecast(steps=1, exog=next_x)
    arimax_preds.append(fc.predicted_mean[0])
    history_y.append(test_df[TARGET].iloc[i])
    history_x.append(test_df[EXOG_COLS].iloc[i].tolist())
    if (i + 1) % 50 == 0:
        print(f"   … {i+1}/{len(test_df)} steps")

arimax_preds = np.array(arimax_preds)
arimax_rmse = np.sqrt(mean_squared_error(y_true, arimax_preds))
arimax_mae  = mean_absolute_error(y_true, arimax_preds)
arimax_u    = theils_u(y_true, arimax_preds, rw_preds)
print(f"\n✅ ARIMAX  RMSE={arimax_rmse:.4f}  MAE={arimax_mae:.4f}  Theil's U={arimax_u:.4f}")


## 10. 🧠 Model 3 — LSTM (multivariate)

**Compute note:** a true per-step refit (like ARIMA's) for a 252-day test window would
mean retraining a deep net ~252 times. That's not tractable here, so we use the
standard practitioner compromise: train once, then forecast **one step at a time
using real (not predicted) lagged inputs at every step** — a genuine walk-forward
*forecast*, with **5 periodic retrains** across the test window to approximate
rolling-origin re-estimation without the full cost. This is stated explicitly so the
comparison's caveats are transparent.


In [ ]:

LOOKBACK = 60
feature_cols = [TARGET] + EXOG_COLS

scaler = MinMaxScaler()
scaled = scaler.fit_transform(data[feature_cols])
target_scaler = MinMaxScaler()
target_scaler.fit(data[[TARGET]])

def make_sequences(arr, lookback):
    X, y = [], []
    for i in range(lookback, len(arr)):
        X.append(arr[i - lookback:i])
        y.append(arr[i, 0])
    return np.array(X), np.array(y)

X_all, y_all = make_sequences(scaled, LOOKBACK)
seq_dates = data.index[LOOKBACK:]

# align split so the LSTM test window == the same TEST_DATES as ARIMA
test_start_pos = np.searchsorted(seq_dates, TEST_DATES[0])
X_train_full, y_train_full = X_all[:test_start_pos], y_all[:test_start_pos]
X_test, y_test = X_all[test_start_pos:], y_all[test_start_pos:]
val_cut = int(len(X_train_full) * 0.85)
X_train, y_train = X_train_full[:val_cut], y_train_full[:val_cut]
X_val, y_val = X_train_full[val_cut:], y_train_full[val_cut:]

print(f"Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}")
assert len(X_test) == len(TEST_DATES), "LSTM test window must match ARIMA test window"


In [ ]:

def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
    return model

model = build_lstm((LOOKBACK, len(feature_cols)))
model.summary()


In [ ]:

def train_model(Xtr, ytr, Xvl, yvl, epochs=60):
    m = build_lstm((LOOKBACK, len(feature_cols)))
    cb = [EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)]
    h = m.fit(Xtr, ytr, validation_data=(Xvl, yvl), epochs=epochs, batch_size=32, callbacks=cb, verbose=0)
    return m, h

model, train_history = train_model(X_train, y_train, X_val, y_val, epochs=30)
print(f"✅ Initial training done. Final val_loss={train_history.history['val_loss'][-1]:.6f}")


In [ ]:

plt.figure(figsize=(9, 4))
plt.plot(train_history.history['loss'], label='train_loss')
plt.plot(train_history.history['val_loss'], label='val_loss')
plt.title('LSTM Training Curve'); plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)'); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:

# Walk-forward LSTM forecast with periodic retraining (5 retrains across the test window)
N_RETRAIN = 3
retrain_every = max(1, len(X_test) // N_RETRAIN)
lstm_preds_scaled = np.zeros(len(X_test))

cur_model = model
cur_Xtr, cur_ytr = X_train_full.copy(), y_train_full.copy()

for i in range(len(X_test)):
    lstm_preds_scaled[i] = cur_model(X_test[i:i+1], training=False).numpy()[0, 0]
    if (i + 1) % retrain_every == 0 and (i + 1) < len(X_test):
        cur_Xtr = np.concatenate([cur_Xtr, X_test[max(0, i - retrain_every + 1):i + 1]])
        cur_ytr = np.concatenate([cur_ytr, y_test[max(0, i - retrain_every + 1):i + 1]])
        vcut = int(len(cur_Xtr) * 0.9)
        cur_model, _ = train_model(cur_Xtr[:vcut], cur_ytr[:vcut], cur_Xtr[vcut:], cur_ytr[vcut:], epochs=15)
        print(f"   🔁 retrained at step {i+1}/{len(X_test)}")

lstm_preds = target_scaler.inverse_transform(lstm_preds_scaled.reshape(-1, 1)).flatten()
y_test_actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

lstm_rmse = np.sqrt(mean_squared_error(y_test_actual, lstm_preds))
lstm_mae  = mean_absolute_error(y_test_actual, lstm_preds)
lstm_u    = theils_u(y_test_actual, lstm_preds, rw_preds)
print(f"\n✅ LSTM  RMSE={lstm_rmse:.4f}  MAE={lstm_mae:.4f}  Theil's U={lstm_u:.4f}")


## 11. 🎯 Uncertainty Quantification for the LSTM (MC-Dropout)

ARIMA gives confidence intervals natively. For the LSTM, we run **MC-Dropout**: keep
dropout active at inference time and run many stochastic forward passes — the spread
of outputs approximates a predictive distribution.

In [ ]:

def mc_dropout_predict(m, X, n_passes=100):
    f = tf.function(lambda x: m(x, training=True))
    preds = np.stack([f(X).numpy().flatten() for _ in range(n_passes)], axis=0)
    return preds  # shape (n_passes, n_samples)

mc_preds_scaled = mc_dropout_predict(cur_model, X_test, n_passes=40)
mc_preds = target_scaler.inverse_transform(mc_preds_scaled.reshape(-1, 1)).reshape(mc_preds_scaled.shape)
lstm_mean = mc_preds.mean(axis=0)
lstm_lo   = np.percentile(mc_preds, 2.5, axis=0)
lstm_hi   = np.percentile(mc_preds, 97.5, axis=0)
print(f"✅ MC-Dropout complete. Mean 95% interval width = {np.mean(lstm_hi - lstm_lo):.4f} pp")


## 12. 📊 Forecast Comparison Chart (with uncertainty bands)

In [ ]:

plt.figure(figsize=(13, 6))
plt.plot(TEST_DATES, y_true, color='black', lw=1.6, label='Actual Breakeven')
plt.plot(TEST_DATES, arima_preds, color='#1f77b4', lw=1.1, label='ARIMA')
plt.fill_between(TEST_DATES, arima_ci_lo, arima_ci_hi, color='#1f77b4', alpha=0.15)
plt.plot(TEST_DATES, arimax_preds, color='#2ca02c', lw=1.1, label='ARIMAX')
plt.plot(TEST_DATES, lstm_mean, color='#d62728', lw=1.1, label='LSTM (MC-dropout mean)')
plt.fill_between(TEST_DATES, lstm_lo, lstm_hi, color='#d62728', alpha=0.15)
plt.plot(TEST_DATES, rw_preds, color='gray', lw=0.9, ls='--', label='Random Walk')
plt.legend(); plt.title('Model Comparison — Test Period Forecasts'); plt.ylabel('Breakeven Inflation (%)')
plt.tight_layout(); plt.show()


In [ ]:

errors = pd.DataFrame({
    'ARIMA':  y_true - arima_preds,
    'ARIMAX': y_true - arimax_preds,
    'LSTM':   y_true - lstm_mean,
    'Random Walk': y_true - rw_preds,
})
fig, ax = plt.subplots(figsize=(10, 5))
for col in errors.columns:
    sns.kdeplot(errors[col], ax=ax, label=col, lw=1.6)
ax.axvline(0, color='black', lw=0.8, ls=':')
ax.set_title('Forecast Error Distributions'); ax.legend()
plt.tight_layout(); plt.show()


## 13. 🧪 Diebold-Mariano Tests (statistical significance)

Numerically lower RMSE doesn't mean a model is *significantly* better. The
Diebold-Mariano test (with the Harvey-Leybourne-Newbold small-sample correction)
tests whether the difference in forecast-error loss between two models is
distinguishable from noise.

In [ ]:

def dm_test(e1, e2, h=1, power=2):
    # Diebold-Mariano test with HLN small-sample correction.
    # e1, e2: forecast error series for model 1 and model 2.
    # Returns (DM_stat, p_value). Negative stat => model 1 has lower loss.
    e1, e2 = np.asarray(e1), np.asarray(e2)
    d = np.abs(e1) ** power - np.abs(e2) ** power
    n = len(d)
    d_bar = d.mean()

    def autocov(d, lag):
        return np.sum((d[lag:] - d_bar) * (d[:n - lag] - d_bar)) / n

    gamma0 = autocov(d, 0)
    var_d = gamma0
    for lag in range(1, h):
        var_d += 2 * autocov(d, lag)
    var_d /= n
    if var_d <= 0:
        return 0.0, 1.0
    dm_stat = d_bar / np.sqrt(var_d)

    hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    dm_stat_adj = dm_stat * hln
    p_value = 2 * (1 - stats.t.cdf(np.abs(dm_stat_adj), df=n - 1))
    return dm_stat_adj, p_value

pairs = [
    ('ARIMA', 'ARIMAX', errors['ARIMA'], errors['ARIMAX']),
    ('ARIMAX', 'LSTM', errors['ARIMAX'], errors['LSTM']),
    ('LSTM', 'Random Walk', errors['LSTM'], errors['Random Walk']),
    ('ARIMA', 'Random Walk', errors['ARIMA'], errors['Random Walk']),
]
dm_rows = []
for name1, name2, e1, e2 in pairs:
    stat, pval = dm_test(e1, e2)
    sig = '✅ significant' if pval < 0.05 else '❌ not significant'
    better = name1 if stat < 0 else name2
    dm_rows.append({'Model A': name1, 'Model B': name2, 'DM Stat': round(stat, 3),
                     'p-value': round(pval, 4), 'Lower loss': better, 'Significant @5%': sig})

dm_table = pd.DataFrame(dm_rows)
display(dm_table)


## 14. 🧭 Directional Accuracy

In [ ]:

from sklearn.metrics import confusion_matrix

actual_dir = np.sign(np.diff(np.concatenate([[data[TARGET].iloc[split_idx - 1]], y_true])))

def directional_acc(preds, actual_dir, name):
    pred_dir = np.sign(np.diff(np.concatenate([[data[TARGET].iloc[split_idx - 1]], preds])))
    acc = (pred_dir == actual_dir).mean()
    cm = confusion_matrix(actual_dir, pred_dir, labels=[-1, 0, 1])
    return acc, cm

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
dir_results = {}
for ax, (name, preds) in zip(axes, [('ARIMA', arima_preds), ('ARIMAX', arimax_preds),
                                      ('LSTM', lstm_mean), ('Random Walk', rw_preds)]):
    acc, cm = directional_acc(preds, actual_dir, name)
    dir_results[name] = acc
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Down','Flat','Up'], yticklabels=['Down','Flat','Up'], ax=ax)
    ax.set_title(f'{name}\nHit-rate={acc:.1%}'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()
print(dir_results)


## 15. 🏦 Benchmark vs. an Institutional Forecast (stretch goal — not implemented)

The Cleveland Fed Inflation Nowcast and NY Fed Survey of Consumer Expectations are
both public, but **both are served from domains outside this environment's network
whitelist** and returned no response when checked. This comparison is left out rather
than faked. **To complete it locally:** pull the Cleveland Fed nowcast CSV from
`https://www.clevelandfed.org/indicators-and-data/inflation-nowcasting` or the NY Fed
SCE from `https://www.newyorkfed.org/microeconomics/sce`, align dates, and add a
`Institutional Forecast` column to the `errors` / scorecard tables above.

## 16. 💰 Simple Trading Application (stretch goal)

Rule: if the LSTM's forecast implies breakeven inflation will **rise** vs. its own
prior forecast → long a TIP-proxy (here, the synthetic breakeven series itself, scaled,
stands in for a TIPS total-return proxy since we don't have the real ETF price);
otherwise hold cash. Benchmarked against buy-and-hold.

In [ ]:

tip_proxy_ret = pd.Series(y_true, index=TEST_DATES).pct_change().fillna(0).values
signal = np.sign(np.diff(np.concatenate([[lstm_mean[0]], lstm_mean])))  # forecast momentum
strategy_ret = signal * tip_proxy_ret
strategy_equity = (1 + strategy_ret).cumprod()
bh_equity = (1 + tip_proxy_ret).cumprod()

def sharpe(rets, periods=252):
    if rets.std() == 0:
        return 0.0
    return np.sqrt(periods) * rets.mean() / rets.std()

strat_sharpe = sharpe(pd.Series(strategy_ret))
bh_sharpe = sharpe(pd.Series(tip_proxy_ret))

plt.figure(figsize=(11, 5))
plt.plot(TEST_DATES, strategy_equity, label=f'LSTM-signal strategy (Sharpe={strat_sharpe:.2f})')
plt.plot(TEST_DATES, bh_equity, label=f'Buy & Hold (Sharpe={bh_sharpe:.2f})', ls='--')
plt.legend(); plt.title('Trading Backtest — Equity Curves (illustrative, synthetic proxy)')
plt.tight_layout(); plt.show()

print(f"Strategy Sharpe: {strat_sharpe:.3f}   Buy & Hold Sharpe: {bh_sharpe:.3f}")


## 17. 🔮 30-Day Future Forecast

In [ ]:

FUTURE_DAYS = 30
final_arima = ARIMA(data[TARGET], order=(p, d, q)).fit()
fc = final_arima.get_forecast(steps=FUTURE_DAYS)
arima_future_mean = fc.predicted_mean
arima_future_ci = fc.conf_int(alpha=0.05)
future_dates = pd.bdate_range(start=data.index[-1] + timedelta(days=1), periods=FUTURE_DAYS)

last_window = scaled[-LOOKBACK:].copy()
lstm_future_mc = []
for _ in range(30):
    window = last_window.copy()
    path = []
    for _ in range(FUTURE_DAYS):
        inp = window[np.newaxis, :, :]
        pred = cur_model(inp, training=True).numpy()[0, 0]
        path.append(pred)
        new_row = window[-1].copy(); new_row[0] = pred
        window = np.vstack([window[1:], new_row])
    lstm_future_mc.append(path)
lstm_future_mc = np.array(lstm_future_mc)
lstm_future_scaled_mean = lstm_future_mc.mean(axis=0)
lstm_future_inv = target_scaler.inverse_transform(lstm_future_scaled_mean.reshape(-1,1)).flatten()
lstm_future_lo = target_scaler.inverse_transform(np.percentile(lstm_future_mc, 2.5, axis=0).reshape(-1,1)).flatten()
lstm_future_hi = target_scaler.inverse_transform(np.percentile(lstm_future_mc, 97.5, axis=0).reshape(-1,1)).flatten()

print(f"ARIMA Day-30 forecast: {arima_future_mean.iloc[-1]:.3f}%   LSTM Day-30 forecast: {lstm_future_inv[-1]:.3f}%")
print(f"Current breakeven: {data[TARGET].iloc[-1]:.3f}%")


In [ ]:

plt.figure(figsize=(13, 6))
plt.plot(data.index[-180:], data[TARGET].iloc[-180:], color='black', label='Historical Breakeven')
plt.plot(future_dates, arima_future_mean, color='#1f77b4', label='ARIMA Forecast')
plt.fill_between(future_dates, arima_future_ci.iloc[:,0], arima_future_ci.iloc[:,1], color='#1f77b4', alpha=0.15)
plt.plot(future_dates, lstm_future_inv, color='#d62728', label='LSTM Forecast (MC mean)')
plt.fill_between(future_dates, lstm_future_lo, lstm_future_hi, color='#d62728', alpha=0.15)
plt.legend(); plt.title('30-Day Forward Forecast'); plt.tight_layout(); plt.show()


## 18. 📋 Final Scorecard & Conclusions

In [ ]:

scorecard = pd.DataFrame({
    'Model': ['ARIMA', 'ARIMAX', 'LSTM', 'Random Walk'],
    'RMSE':  [arima_rmse, arimax_rmse, lstm_rmse, rw_rmse],
    'MAE':   [arima_mae, arimax_mae, lstm_mae, rw_mae],
    "Theil's U": [arima_u, arimax_u, lstm_u, 1.0],
    'Directional Acc.': [dir_results['ARIMA'], dir_results['ARIMAX'], dir_results['LSTM'], dir_results['Random Walk']],
}).round(4)
display(scorecard)
print()
display(dm_table)


### Conclusion (plain language)

- **Beat-the-naive test:** any model with **Theil's U < 1** outperformed the random
  walk; check the scorecard above — if a model's U is ≥ 1, it is **not** earning its
  complexity and that should be stated honestly rather than spun.
- **Statistical significance:** the DM test table is the real arbiter. A model with a
  lower RMSE *and* a DM p-value < 0.05 against the next-best model has a defensible
  edge. A lower RMSE with p > 0.05 means the apparent edge is **not distinguishable
  from noise** on this sample — report it as a tie, not a win.
- **ARIMAX vs LSTM:** because both see the same exogenous information set, this is the
  fairest single comparison in the notebook. If ARIMAX is statistically indistinguishable
  from the LSTM, that's a legitimate and common finding in short macro time series —
  the LSTM's extra flexibility doesn't pay for itself when the sample is this size, and
  a Street audience would respect that conclusion more than an inflated complexity claim.
- **Directional accuracy** matters more than RMSE for a trading desk: a model can have
  a mediocre RMSE but a useful hit-rate above 55%, which is what the Section 16 backtest
  Sharpe ratio is trying to capture.
- **Caveat to read every time this notebook is reused:** all data here is a calibrated
  **synthetic proxy**, not live FRED/Yahoo data (network restricted in this build
  environment — see Section 2 for the live swap-in code). Treat every number as a
  *demonstration of methodology*, not a live market call, until rerun with real data.
